In [ ]:
import sys
import os
from pathlib import Path

pwd = Path().resolve()
sys.path.append(str(pwd.parent))
sys.path.append(str(pwd / 'src'))
sys.path.append(str(pwd / 'hacv2026' / 'src'))
%load_ext autoreload
%autoreload 2

/home/mareko/Source/SpOC3/Challenge 3 Programmable Cubes/solution/hacv2026/src


In [2]:
%matplotlib inline
import numpy as np
import numpy.typing as npt
from tqdm import tqdm
from programmable_cubes_UDP import ProgrammableCubes
from programmable_cubes_UDP import programmable_cubes_UDP
from src.implementation_heuristic import *

# Import from https://gitlab.fjfi.cvut.cz/ksi/hacv-2026
from objfun_tsp import TSPGrid
from heur_fsa import FastSimulatedAnnealing
from heur_aux import CauchyMutation, Correction
from objfun import ObjFun


DEBUG = False

## Info

This notebook is for testing general heuristics on the problem programmable_cubes_UDP with no task-specific knowledge apart from A* implementation on the cubeset.



## Load

In [ ]:
PROBLEM = "ISS"
udp = programmable_cubes_UDP(PROBLEM,"../")
udp.fitness(np.array([-1]))
ti = udp.initial_cube_types
ci = udp.final_cube_positions
ct = udp.target_cube_positions
tt = udp.target_cube_types
types = np.arange(np.max(ti)+1)

## Functions to decode solution and compute fitness function

In [ ]:
def chromosome_from_pairs(udp:programmable_cubes_UDP,ids_to_move,ids_to_fill,wrong_penalty,budget):
    """ 
    cubes
    wi - ids of cubes that are wrong
    ct - target coords 
    
    """
    fitness = 0
    # ti = udp.initial_cube_types
    # types = np.arange(np.max(ti)+1)
    targets = udp.target_cube_positions

    cubes = ProgrammableCubes(udp.final_cube_positions)
    chrom = []
    for id_to_move,id_to_fill in zip(ids_to_move,ids_to_fill):
        id = id_to_move
        end = targets[id_to_fill]
        if not is_connected(end,cubes.cube_position):
            continue

        #tmp_chrom, success = axis_search(cubes,id,end)
        #if not success:
        tmp_chrom, success = astar_cubes(cubes,id,end,budget)
        if success:
            chrom.extend(tmp_chrom)
            cubes.apply_chromosome(np.concatenate([tmp_chrom,[-1]]),True)
        else:
            #break
            fitness += wrong_penalty

    #print( udp.fitness([-1],cubes.cube_position))
    fitness += udp.fitness(np.array([-1]),cubes.cube_position)[0]
    # udp.plot('ensemble', types)
    # udp.plot('target',types)
    return np.array(chrom,dtype=int),fitness

def double_lehmer_decode(a,lehmer):
    """
    unpacks the even dimensional solution in a from lehmer code
    """
    length = len(a)
    half_length = int(length/2)
    return np.concatenate(
        [lehmer_decode(a[:half_length],lehmer[:half_length]),
         lehmer_decode(a[half_length:length],lehmer[half_length:length])])

def lehmer_decode(arr,lehmer):
    """
    unpacks the dimensional solution in a from lehmer code
    """
    arr_copy = list(arr)
    res = np.zeros(shape=len(arr),dtype=int)
    for id,val in enumerate(lehmer):
        res[id] = arr_copy[val]
        del arr_copy[val]
    return res

def split_array_to_halves(arr):
    half_length = int(len(arr)/2)
    return arr[:half_length], arr[half_length:len(arr)]

if DEBUG:
    wii, wit = get_wrong_ids_and_coords(udp)
    x = np.concatenate([wii,wit])
    size = len(wii)
    #lehmer = np.zeros(size,dtype=int)
    lehmer = np.zeros(size*2,dtype=int)
    lehmer[1] = 0
    #print(lehmer_decode(wii,lehmer))
    print(wii,wit)
    print(split_array_to_halves(double_lehmer_decode(x,lehmer)))




## Objective function definition

- uses types from https://gitlab.fjfi.cvut.cz/ksi/hacv-2026

In [ ]:


class Cubes(ObjFun):
    udp : programmable_cubes_UDP
    n : int
    wii : np.ndarray # wrong ids of the initial config
    wit : np.ndarray # wrong ids of the target config
    all_wi : np.ndarray
    budget : int = 100

    def __init__(self, udp : programmable_cubes_UDP) -> None:
        """

                """
        # Get the domain, find out amount of errors and create "double lehmer domain"
        wii, wit = get_wrong_ids_and_coords(udp) 
        # TODO: ordering is randomized from structure ProgrammableCubes
        self.wii = wii
        self.wit = wit
        self.all_wi = np.concatenate([wii,wit])

        self.n = len(wii)+len(wit) # dimension
        half_n = len(wii)

        self.udp = udp
        self.fstar = -1.



        self.a = np.zeros(self.n, dtype=np.int64)
        lehmer_upper_bound_half = np.arange(half_n - 1, -1, -1, dtype=np.int64)
        self.b = np.concatenate([lehmer_upper_bound_half,lehmer_upper_bound_half])

    def generate_point(self, rng: np.random.Generator = None) -> npt.NDArray[np.int64]:
        """
        
        """
        

        return np.zeros(shape=self.n,dtype=int) # basic same initialization

    def evaluate(self, x: npt.NDArray[np.int64], wrong_penalty = 1) -> np.float64:
        perm_wii, perm_wit = split_array_to_halves(double_lehmer_decode(self.all_wi,x))
        self.udp.fitness(np.array([-1]))
        chrom, fitness = chromosome_from_pairs(self.udp,perm_wii,perm_wit,wrong_penalty,self.budget)

        return fitness

    def get_neighborhood(self, x: npt.NDArray[np.int64], d: int) -> list[npt.NDArray[np.int64]]:
        """
        Returns all solutions reachable by incrementing or decrementing one
        component of x by 1 (diameter-1 neighbourhood in the Lehmer encoding).
        """
        assert d == 1, "supports neighbourhood with distance = 1 only"
        nd = []
        for i, xi in enumerate(x):
            if x[i] > self.a[i]:
                xl = x.copy(); xl[i] -= 1; nd.append(xl)
            if x[i] < self.b[i]:
                xu = x.copy(); xu[i] += 1; nd.append(xu)
        return nd

In [ ]:
# Declare the problem
problem = Cubes(udp)

if DEBUG:
    # Create a point and compute fitness for it
    point = problem.generate_point()
    print(problem.evaluate(point,0),point)

## Method

In [ ]:

def run_fsa(problem, maxeval, T0, n0, alpha, r, label = "none"):
    records = []
    mut = CauchyMutation(r=r, correction=Correction(problem))
    res = FastSimulatedAnnealing(problem, maxeval=maxeval, T0=T0, n0=n0, alpha=alpha,
                                 mutation=mut, seed=0).search()
    records.append({'label': label, 'seed': 0,
                    'best_y': res['best_y'], 'neval': res['neval'],
                    'success': res['success'], 'best_x': res['best_x']})
    return records



## Run

In [ ]:
res = run_fsa(problem,10,1,5,2,2)

## Results

In [ ]:

print("Fitness for problem: ",problem.evaluate(res[0]["best_x"],0))

print("Fitness with added error values: ",problem.evaluate(res[0]["best_x"],1))